# Interactive Jupyter Visualization via `TmapViz`

This notebook uses the same quickstart-style TMAP pipeline, but configures visualization through `TmapViz` and then renders either in-notebook (`to_jupyter`) or as HTML (`save`).


## 1. Imports


In [1]:
from importlib.metadata import PackageNotFoundError, version
try:
    print("jupyter-scatter:", version("jupyter-scatter"))
except PackageNotFoundError:
    print("jupyter-scatter is not installed")


jupyter-scatter: 0.22.2


## 2. Create fingerprints and calculate some properties 
You don'really need to worry about this part.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# Check RDKit availability
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

from tmap import LSHForest, MinHash
from tmap.layout import LayoutConfig, ScalingType, layout_from_lsh_forest
from tmap.visualization import TmapViz


def smiles_to_fingerprints(
    smiles_list: list[str],
    radius: int = 2,
    n_bits: int = 2048,
) -> tuple[np.ndarray, list[int], list]:
    """
    Convert SMILES strings to Morgan fingerprints.

    Args:
        smiles_list: List of SMILES strings
        radius: Morgan fingerprint radius (default 2 = ECFP4)
        n_bits: Number of bits in fingerprint

    Returns:
        fingerprints: numpy array of shape (n_valid, n_bits)
        valid_indices: indices of successfully parsed SMILES
        mols: list of RDKit molecule objects
    """
    if not RDKIT_AVAILABLE:
        raise ImportError("RDKit is required: pip install rdkit")

    # Use the newer MorganGenerator API
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    fingerprints = []
    valid_indices = []
    mols = []

    for i, smi in tqdm(enumerate(smiles_list),
                       desc='Generating Fingerprints', total=len(smiles_list)):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = morgan_gen.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp.astype(np.uint8))
            valid_indices.append(i)
            mols.append(mol)
    return np.array(fingerprints), valid_indices, mols


def compute_molecular_properties(mols: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute molecular properties for coloring."""
    mw = [Descriptors.MolWt(mol) for mol in mols] # type: ignore
    logp = [Descriptors.MolLogP(mol) for mol in mols] # type: ignore
    n_rings = [rdMolDescriptors.CalcNumRings(mol) for mol in mols]
    return np.array(mw), np.array(logp), np.array(n_rings)

df = pd.read_csv('/Users/afloresep/work/tmap2/examples/cluster_65053.csv')

fingerprints, valid_indices, mols = smiles_to_fingerprints(df.smiles.to_list())
mw, logp, n_rings = compute_molecular_properties(mols)
print(f"fingerprints shape: {fingerprints.shape}")
print(f"average active bits: {fingerprints.sum(axis=1).mean():.1f}")


Generating Fingerprints: 100%|██████████| 64098/64098 [00:05<00:00, 12480.05it/s]


fingerprints shape: (64098, 2048)
average active bits: 47.0


## 3. Encode + index + compute layout


There's two ways to do this part. One via the old TMAP API whihc remains the same but is more verbose (MinHash -> LSHForest -> layout_from*). The alternative is the new `TMAP` class which allows for a more simpler a la sklearn API 

In [3]:
# The short way 
from tmap import TMAP
tm = TMAP()
x, y, s, t= tm.fit_transform(fingerprints)
print(f"nodes: {len(x)}, edges: {len(s)}")


nodes: 64098, edges: 64097


In [4]:
# Or the long way
mh = MinHash()
signatures = mh.batch_from_binary_array(fingerprints)

lsh = LSHForest()
lsh.batch_add(signatures)
lsh.index()

x, y, s, t = layout_from_lsh_forest(lsh)
n = len(x)

print(f"signature shape: {signatures.shape}")
print(f"nodes: {n}, edges: {len(s)}")


signature shape: (64098, 128)
nodes: 64098, edges: 64097


In [ ]:
# For a quick visualization

viz = TmapViz()
viz.set_points(x,y) # Set x, y coordinates
viz.background_color = "#FFFFFF"
# Add some color layouts -you can later choose between them- 
viz.add_color_layout("Molecular Weight", mw.tolist())
viz.add_color_layout("Number of Rings", categorical=True, values= n_rings.tolist(), color='tab10')
viz.add_color_layout("LogP", logp.tolist(), categorical=False, add_as_label=True)
# Shows in the tooltip
viz.add_label("smiles", df.smiles.tolist()) 
viz.show()


# Widget
If you want to access all functionalities like selection or zoom, then create a widget

In [ ]:
# Basically the same as before, only now we can do much more cool stuff
widget =viz.to_widget()
widget.show()


In [ ]:
# We can try other functionalities like exporting the index of selected clusters!
# First, long press the left click of your mouse and then drag an area to select some points

#We now can call widget.selection() to export the index 
widget.selection()


array([46812, 48225, 18462, 45371, 51431, 45372, 52015, 41415, 51908,
       55664, 30236, 48120, 48231, 51438, 35833, 35832, 35819, 56827,
       56828, 48156, 48079, 44818, 54260, 54159, 48289, 56843, 18778,
       54189, 56812, 48233, 48160, 54281, 30196, 18777, 54188, 54187,
       48262, 48209], dtype=uint32)

## Pandas dataframe
but what good is the index if we don't have a pandas dataframe with the rest of the values we added? 

In [ ]:
df = viz.to_dataframe()
df.head()


,Molecular Weight,Number of Rings,LogP,smiles,_tmap_x,_tmap_y
0,395.837,3,3.6688,C=C1CCN(C(=O)C2=CC=C(Cl)C=N2)C(C(=O)N2CCC(=C(F...,0.552632,-0.768832
1,348.349,3,2.4556,C=C1CCN(C(=O)C2=CC=C(F)C(F)=C2)C(C(=O)N2CC=CCO...,0.254497,-0.798056
2,350.365,3,1.9844,C=C1CCN(C(=O)C2=CC=C(F)C(F)=C2)C(C(=O)N2CCOCC2)C1,0.252418,-0.792052
3,378.469,3,2.6560,C=C1CCN(C(=O)C2=CC=C(F)C(O)=C2)C(C(=O)N2CCSCC2...,0.255245,-0.792482
4,380.485,3,2.6674,C=C1CCN(C(=O)C2=CC=C(F)S2)C(C(=O)N2CCC(O)CC(C)...,0.270149,-0.793297


In [ ]:
# now we can pass the index

df.loc[widget.selection()]


,Molecular Weight,Number of Rings,LogP,smiles,_tmap_x,_tmap_y
46812,336.334,4,2.78920,O=C1C2=CC=CC=C2CC1OC(=O)C1(C2CCOCC2)CC1(F)F,-0.945627,-0.217241
48225,397.903,4,3.33120,O=S(=O)(CC1=CC=C(F)C(Cl)=C1)N1CCC(N2C=NC3=C2CC...,-0.961299,-0.246522
18462,377.416,4,3.51242,CC1=CC=CC(CC2=CN=C(N3C(=O)NC4(CC(C(F)F)C4)C3=O...,-0.948273,-0.241603
45371,404.388,4,4.86830,O=C1NC2(CCC(C(F)F)CC2)C(=O)N1C1=CC=C(OC2=CC(F)...,-0.944110,-0.239918
51431,397.834,4,3.85740,O=C1NC2(CC(C(F)F)C2)C(=O)N1C1=NC=C(CC2=CC(Cl)=...,-0.947118,-0.242998
45372,382.838,4,4.11750,O=C1NC2(CCC(C(F)F)CC2)C(=O)N1CC1(C2=CC(Cl)=CC=...,-0.945352,-0.241221
52015,382.838,4,4.01850,O=C1N[C@H](CC2=CC(Cl)=CC=C2)C(=O)N1CC1CCC2(CC1...,-0.944943,-0.237577
41415,358.825,3,3.20510,O=C1CC(=O)N(C2CCCCC2)C(=O)N1CC#CC1=CC(Cl)=CC=C1,-0.946718,-0.237640
51908,382.838,4,4.01850,O=C1N[C@@H](CC2=CC(Cl)=CC=C2)C(=O)N1CC1CCC2(CC...,-0.948328,-0.239922
55664,382.838,4,4.01850,O=C1NC(CC2=CC(Cl)=CC=C2)C(=O)N1CC1CCC2(CC1)CC2...,-0.947848,-0.238372


# Export the same `TmapViz` config to HTML
We can also export the TMAP to a html file to share or visualize in your browser

In [14]:
output_path = viz.write_html("./")
print("Saved:", output_path)


Saved: MyTMAP.html
